# 04 · FIRMS query plan and external fire evidence
Progress of the harmonization plan: 65 %. This annex does not train or validate a model.
The team's JSON organizes NASA queries; it does not contain independent evidence.
Its structure is audited and a catalog of 2019 COEN–INDECI reports is started.
The events are documented; their match with pixels is still pending.
No detections from the held-out year 2024 are used to fit or select models.

In [1]:
from pathlib import Path
from datetime import date, timedelta
from urllib.parse import urlsplit
import os
import json
import hashlib
import pandas as pd
from IPython.display import display
REPO_ROOT = Path(os.environ.get('FIRMS_REPO_DIR', str(Path.cwd()))).resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
CONFIG = REPO_ROOT / 'server' / 'data' / 'api_links.json'
ARTIFACTS = Path(os.environ.get('FIRMS_ARTIFACTS_DIR', str(REPO_ROOT / 'artifacts')))
OUT = ARTIFACTS / '05_evidencia_externa'
OUT.mkdir(parents=True, exist_ok=True)
plan = json.loads(CONFIG.read_text(encoding='utf-8-sig'))
plan_hash = hashlib.sha256(CONFIG.read_bytes()).hexdigest()

## 1. JSON audit
Date limits, block continuity, inclusive days, host, source, region and path
are checked. The queries are not run, no key is inserted
and remote availability is not checked. The syntactic audit does not guarantee
that each endpoint responds or that the observations are complete.

In [2]:
def audit_period(region, spec, period):
    expected = date.fromisoformat(period['inicio'])
    rows = []
    kind = 'area' if spec['tipo'] == 'area' else 'country'
    for block in period['bloques']:
        start, end = date.fromisoformat(block['inicio']), date.fromisoformat(block['fin'])
        days = (end - start).days + 1
        assert start == expected and days == block['dias'] and 1 <= days <= 5
        assert set(block['urls']) == {'MODIS_SP', 'VIIRS_SNPP_SP'}
        for sensor, url in block['urls'].items():
            parsed = urlsplit(url)
            assert parsed.scheme == 'https' and parsed.hostname == 'firms.modaps.eosdis.nasa.gov'
            bits = parsed.path.strip('/').split('/')
            assert bits == ['api', kind, 'csv', '{KEY}', sensor, spec['area'], str(days), block['inicio']]
        rows.append({'region': region, 'start': str(start), 'end': str(end),
                     'days': days, 'urls': len(block['urls']), 'endpoint': kind})
        expected = end + timedelta(days=1)
    assert expected == date.fromisoformat(period['fin']) + timedelta(days=1)
    return rows

In [3]:
blocks, periods = [], []
for region, spec in plan['regiones'].items():
    if spec['tipo'] == 'area':
        west, south, east, north = map(float, spec['area'].split(','))
        assert -180 <= west < east <= 180 and -90 <= south < north <= 90
    for period in spec['periodos']:
        blocks.extend(audit_period(region, spec, period))
        periods.append({'region': region, 'area': spec['area'], 'usage': spec['uso'],
                        'start': period['inicio'], 'end': period['fin']})
blocks, periods = pd.DataFrame(blocks), pd.DataFrame(periods)
print('Regiones:', len(plan['regiones']), '| períodos:', len(periods),
      '| bloques:', len(blocks), '| URLs:', int(blocks.urls.sum()))
display(periods)
blocks.to_csv(OUT / 'auditoria_bloques.csv', index=False)

Regiones: 4 | períodos: 12 | bloques: 206 | URLs: 412


,region,area,usage,start,end
0,ucayali,"-75.7,-11.5,-70.4,-7.2",aprender y depurar,2019-08-15,2019-09-30
1,cusco_puno,"-73.9,-17.3,-68.8,-11.2",contraste Andes,2019-07-01,2019-09-30
2,nororiente_2024,"-78.8,-11.5,-70.4,-2.9",demo y validacion,2024-09-01,2024-09-30
3,peru,PER,calibrar,2013-08-01,2013-10-31
4,peru,PER,calibrar,2014-08-01,2014-10-31
5,peru,PER,calibrar,2015-08-01,2015-10-31
6,peru,PER,calibrar,2016-08-01,2016-10-31
7,peru,PER,calibrar,2017-08-01,2017-10-31
8,peru,PER,calibrar,2018-08-01,2018-10-31
9,peru,PER,calibrar,2019-08-01,2019-10-31


## 2. How to use it without accidentally changing the evaluation
- Ucayali 2019: exploration and local cases.
- Cusco/Puno 2019: geographic comparison, after pinning down the location of each event.
- Northeast September 2024: can be held as a final illustrative case;
  do not use it to select the model if 2024 is still a held-out test.
- Peru 2013–2021: the JSON only includes August–October. Calibrating only
  with those months would not show that it works during the whole year.

The approximate boxes are not administrative boundaries and may include
territory outside the named region or outside Peru. The existing national download
already contains the 2019 detections: there is no need to duplicate it to
start. Filtering MODIS to Aqua is still necessary.

## 3. Initial catalog of independent evidence
Facts summarized from official documents consulted on 14/11/2026.
They are cases selected for study, not an exhaustive inventory or a random
sample. On their own they do not provide exact coordinates or satellite coverage.
The event date is kept apart from the publication date; no coordinates are invented.

In [4]:
sources = {
    'yarinacocha': 'https://portal.indeci.gob.pe/wp-content/uploads/2019/09/REPORTE-COMPLEMENTARIO-N%C2%BA-2206-19SEP2019-INCENDIO-FORESTAL-EN-EL-DISTRITO-DE-YARINACOCHA-UCAYALI-01.pdf',
    'curimana': 'https://portal.indeci.gob.pe/wp-content/uploads/2019/09/REPORTE-COMPLEMENTARIO-N%C2%BA-2027-06SEP2019-INCENDIO-FORESTAL-EN-EL-DISTRITO-DE-CURIMANA-UCAYALI-02.pdf',
    'iparia': 'https://portal.indeci.gob.pe/emergencias/62829/',
    'macari': 'https://portal.indeci.gob.pe/emergencias/reporte-complementario-n-2289-02-09-2019-coen-indeci-1150-horas-reporte-no-01-incendio-forestal-en-el-distrito-de-acari-puno/',
}
events = pd.DataFrame([
    {'event_id': 'coen_2019_yarinacocha_san_juan', 'json_region': 'ucayali',
     'department': 'Ucayali', 'district': 'Yarinacocha', 'locality': 'San Juan',
     'event_date': '2019-09-14', 'event_start_time_reported': '14:00 aprox.',
     'publication_date': '2019-09-19', 'confirmed_end_local': '2019-09-14 15:00',
     'source_url': sources['yarinacocha'], 'note': 'Duración corta; no presumir paso satelital durante el fuego.'},
    {'event_id': 'coen_2019_curimana_dos_de_mayo', 'json_region': 'ucayali',
     'department': 'Ucayali', 'district': 'Curimaná', 'locality': 'Dos de Mayo',
     'event_date': '2019-08-31', 'event_start_time_reported': '17:30',
     'publication_date': '2019-09-06', 'confirmed_end_local': None,
     'source_url': sources['curimana'], 'note': 'El reporte incluye información municipal y fotografías; precisar perímetro.'},
    {'event_id': 'coen_2019_iparia_el_sira', 'json_region': 'ucayali',
     'department': 'Ucayali', 'district': 'Iparia', 'locality': 'El Sira, cerca de Cunchuri',
     'event_date': '2019-09-08', 'event_start_time_reported': '15:30 aprox.',
     'publication_date': '2019-09-23', 'confirmed_end_local': None,
     'source_url': sources['iparia'], 'note': 'La fecha de publicación no establece el final del incendio.'},
    {'event_id': 'coen_2019_macari_selque_huamanruro', 'json_region': 'cusco_puno',
     'department': 'Puno', 'district': 'Macari', 'locality': 'Selque y Huamanruro',
     'event_date': '2019-09-26', 'event_start_time_reported': '10:30 y 13:00',
     'publication_date': '2019-09-27', 'confirmed_end_local': None,
     'source_url': sources['macari'], 'note': 'Dos localidades/horas: separar subeventos antes de emparejar píxeles.'},
])

In [5]:
events['publication_lag_days'] = (pd.to_datetime(events.publication_date)
                                  - pd.to_datetime(events.event_date)).dt.days
events['source_type'] = 'COEN-INDECI official event report'
events['location_precision'] = 'named locality/district; coordinates not digitized'
events['latitude'] = pd.NA
events['longitude'] = pd.NA
events['satellite_match_status'] = 'pending_georeferencing_and_observation_window'
events['review_status'] = 'documented_event_not_satellite_validated'
assert events.event_id.is_unique and events.publication_lag_days.ge(0).all()
for _, event in events.iterrows():
    allowed = plan['regiones'][event.json_region]['periodos']
    assert any(p['inicio'] <= event.event_date <= p['fin'] for p in allowed)
events.to_csv(OUT / 'catalogo_eventos_documentados.csv', index=False)
events.to_json(OUT / 'catalogo_eventos_documentados.json', orient='records', indent=2, force_ascii=False)
display(events[['district', 'locality', 'event_date', 'publication_date', 'publication_lag_days', 'satellite_match_status']])

,district,locality,event_date,publication_date,publication_lag_days,satellite_match_status
0,Yarinacocha,San Juan,2019-09-14,2019-09-19,5,pending_georeferencing_and_observation_window
1,Curimaná,Dos de Mayo,2019-08-31,2019-09-06,6,pending_georeferencing_and_observation_window
2,Iparia,"El Sira, cerca de Cunchuri",2019-09-08,2019-09-23,15,pending_georeferencing_and_observation_window
3,Macari,Selque y Huamanruro,2019-09-26,2019-09-27,1,pending_georeferencing_and_observation_window


## 4. Comparison protocol: what is missing before saying "validated"
1. Georeference the locality or perimeter with a source and an uncertainty margin.
2. Check the real interval of the fire; use the date of the event, not of the news.
   If Peru local time is assumed, record that assumption and convert it to UTC.
3. Fix space and time tolerances before looking at the hotspots of the case.
4. Look for the original detections of each sensor, keep multiple candidates
   and check coverage/clouds/overpasses when possible.
5. Classify: spatially/temporally compatible; undetermined; or no reported
   detection. Do not automatically mark a false alarm or a confirmed omission.
6. Repeat after harmonizing to check that known episodes are preserved.

A one-hour fire may not coincide with a satellite overpass. No news
is not evidence of absence of fire. A news item that only reproduces FIRMS does not
provide independent validation. A hotspot is not the same as a reported fire.

These four cases do not allow estimating global accuracy or proving that a
correction between sensors works. The method's held-out time evaluation
and the documentary comparison of events are complementary tests.

In [6]:
audit = {
    'harmonization_progress_percent': 65,
    'json_sha256': plan_hash, 'regions': len(plan['regiones']),
    'periods': len(periods), 'blocks': len(blocks), 'urls': int(blocks.urls.sum()),
    'syntax_calendar_audit_passed': True, 'remote_api_requests_executed': False,
    'independent_documented_candidates': len(events), 'satellite_validated_events': 0,
    'event_source_access_date': '2026-11-14',
    'test_year_data_inspected_for_event_matching': False,
    'training_or_harmonization_performed': False,
}
(OUT / 'auditoria_evidencia.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
print('Auditoría completada. Eventos documentados: 4. Coincidencias satelitales verificadas: 0.')

Auditoría completada. Eventos documentados: 4. Coincidencias satelitales verificadas: 0.
